# Defending with the active subspace + why one class attracts the attack

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/defense_and_analysis.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The first notebook showed that the active-subspace universal attack is much stronger than UAP for small perturbations.
Here I look at the other side:

1. **Input-space active subspace**: the top-k eigenvectors $U$ of $\mathbb{E}[\nabla_x \ell \nabla_x \ell^T]$.
   How much of each universal perturbation lies in $\text{span}(U)$?
2. **Purification**: classify $x - UU^T(x-\mu)$ instead of $x$. Cost on clean accuracy vs. gain in robustness,
   also against an **adaptive** attack that knows the defense.
3. **Detection**: score $s(x) = \|U^T(x-\mu)\|^2$, ROC AUC clean vs. attacked (with random noise as a control).
4. **The attractor class**: almost all fooled images went to *erythroblast*. Is it the class closest to the others,
   or does the perturbation look like an erythroblast?
5. Repeat the attacks with 3 random training sets to see if the numbers are stable.

Use a T4 GPU. Total time about 40–50 min (training included).

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from medmnist import INFO

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, evaluate, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline
from asnet.defense import input_active_subspace, Purified, detection_scores, roc_auc, overlap
from asnet import analysis

set_seed(0)
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
class_names = list(INFO["bloodmnist"]["label"].values())
short = [n.split(" (")[0][:14] for n in class_names]
print(device)

## Settings

In [ ]:
SIZE = 64
EPOCHS = 20
KS = [5, 10, 20, 50]     # dimensions of the input active subspace removed / used
AS_SAMPLES = 2000
N_ATTACK = 200
DELTAS = [4, 6]          # where the two attacks differed most
SEEDS = [0, 1, 2]

## Model\n\nReuses the checkpoint from the first notebook if it is in this folder, otherwise trains VGG-19 again.

In [ ]:
train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
model = build_model("vgg19", NUM_CLASSES, SIZE)

if os.path.exists("vgg19_bloodmnist.pt"):
    model.load_state_dict(torch.load("vgg19_bloodmnist.pt", map_location="cpu"))
    model.to(device)
else:
    model, _ = train_classifier(model, make_loader(train_aug, 128, shuffle=True), make_loader(val, 256),
                                epochs=EPOCHS, device=device)
    torch.save(model.state_dict(), "vgg19_bloodmnist.pt")
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

Xte, yte = stack_dataset(test, device=device)
print("test acc", (predict(model, Xte) == yte).float().mean().item())

## 1. Input-space active subspace

In [ ]:
sub = input_active_subspace(model, make_loader(subset(train, AS_SAMPLES, seed=10), 64),
                            k=max(KS), device=device)
lam = sub.eigvals.numpy()[:150]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.semilogy(np.arange(1, len(lam) + 1), lam / lam[0])
a1.set(xlabel="index i", ylabel=r"$\lambda_i/\lambda_1$", title="input-space spectrum")
a2.plot(np.arange(1, len(lam) + 1), np.cumsum(lam) / lam.sum())
for k in KS:
    a2.axvline(k, c="gray", lw=0.8, ls=":")
a2.set(xlabel="k", ylabel="captured energy", title="cumulative energy")
plt.tight_layout(); plt.savefig("input_spectrum.png", bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, min(6, sub.U.shape[1]), figsize=(13, 2.5))
for i, ax in enumerate(axes):
    u = sub.U[:, i].view(3, SIZE, SIZE).permute(1, 2, 0).cpu().numpy()
    ax.imshow((u - u.min()) / (u.max() - u.min())); ax.set_title(f"u{i + 1}"); ax.axis("off")
plt.suptitle("leading input active directions"); plt.tight_layout(); plt.show()

## 2. Attacks with 3 different training sets

In [ ]:
def all_subspace(k):
    s = copy.copy(sub)
    s.U = sub.U[:, :k]
    return s

perts, rows = {}, []
for seed in SEEDS:
    Xtr, ytr = stack_dataset(subset(train, N_ATTACK, seed=100 + seed), device=device)
    for d in DELTAS:
        v_as = as_universal_attack(model, Xtr, ytr, delta=d, seed=seed, log=lambda *a: None).v
        v_uap = uap_baseline(model, Xtr, NUM_CLASSES, delta=d, max_iter=10, device=device)
        for name, v in [("AS", v_as), ("UAP", v_uap)]:
            perts[(seed, d, name)] = v
            rows.append({"seed": seed, "delta": d, "method": name,
                         "test fooling": fooling_rate(model, Xte, v),
                         **{f"in span(U{k})": overlap(v, all_subspace(k)) for k in KS}})
        print(f"seed {seed} delta {d}: AS {rows[-2]['test fooling']:.3f}  UAP {rows[-1]['test fooling']:.3f}")

runs = pd.DataFrame(rows)
runs.groupby(["delta", "method"]).agg(["mean", "std"]).drop(columns="seed").round(3)

## 3. Purification

`transfer`: the perturbation was built on the undefended model and then given to the defended one.
`adaptive`: the attack is run again directly on the defended model (it sees the projection), which is the honest test.

In [ ]:
Xtr, ytr = stack_dataset(subset(train, N_ATTACK, seed=100), device=device)
d = DELTAS[0]
prow = []
for k in [0] + KS:
    net = model if k == 0 else Purified(model, all_subspace(k)).to(device).eval()
    row = {"k": k, "clean acc": (predict(net, Xte) == yte).float().mean().item()}
    for name in ["AS", "UAP"]:
        row[f"{name} transfer"] = fooling_rate(net, Xte, perts[(0, d, name)])
    row["AS adaptive"] = fooling_rate(net, Xte, as_universal_attack(net, Xtr, ytr, delta=d, log=lambda *a: None).v)
    row["UAP adaptive"] = fooling_rate(net, Xte, uap_baseline(net, Xtr, NUM_CLASSES, delta=d, device=device))
    prow.append(row)
    print(row)
purif = pd.DataFrame(prow)
purif.round(3)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(purif.k, purif["clean acc"] * 100, "o-", c="k")
a1.set(xlabel="k (directions removed)", ylabel="%", title="clean test accuracy")
for col, st in [("AS transfer", "o-"), ("UAP transfer", "s-"), ("AS adaptive", "o--"), ("UAP adaptive", "s--")]:
    a2.plot(purif.k, purif[col] * 100, st, label=col)
a2.set(xlabel="k (directions removed)", ylabel="%", title=f"test fooling rate, delta = {d}")
a2.legend()
plt.tight_layout(); plt.savefig("purification.png", bbox_inches="tight"); plt.show()

## 4. Detection

In [ ]:
g = torch.Generator().manual_seed(0)
drow = []
for k in KS:
    s = all_subspace(k)
    clean = detection_scores(s, Xte)
    row = {"k": k}
    for name in ["AS", "UAP"]:
        v = perts[(0, d, name)]
        row[name] = roc_auc(clean, detection_scores(s, (Xte + v).clamp(-1, 1)))
    noise = torch.randn(Xte[:1].shape, generator=g).to(device)
    noise = noise / noise.norm() * d
    row["random noise"] = roc_auc(clean, detection_scores(s, (Xte + noise).clamp(-1, 1)))
    drow.append(row)
det = pd.DataFrame(drow).set_index("k")
print("ROC AUC (1.0 = perfect detection, 0.5 = chance)")
det.round(3)

## 5. The attractor class

Checks, for each perturbation:
- **where** the fooled images go,
- the **runner-up** class of clean images (the closest decision region),
- the average logit **margin** to each class,
- the cosine between $v$ and the **average image of each class** (minus the mean image),
- the cosine between $v$ and the **mean gradient** that increases each class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, name in zip(axes, ["AS", "UAP"]):
    width = 0.8 / len(SEEDS)
    for i, seed in enumerate(SEEDS):
        h = analysis.target_histogram(model, Xte, perts[(seed, DELTAS[-1], name)], NUM_CLASSES).float()
        ax.bar(np.arange(NUM_CLASSES) + i * width, h / h.sum() * 100, width, label=f"seed {seed}")
    ax.set_xticks(np.arange(NUM_CLASSES) + 0.4, short, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{name}: class of fooled images (delta = {DELTAS[-1]})")
axes[0].set_ylabel("% of fooled images"); axes[0].legend()
plt.tight_layout(); plt.savefig("targets.png", bbox_inches="tight"); plt.show()

In [ ]:
v = perts[(0, DELTAS[-1], "AS")]
runner = analysis.runner_up_histogram(model, Xte, NUM_CLASSES).float()
margins = [analysis.margin_to_class(model, Xte, c) for c in range(NUM_CLASSES)]
sim_mean = analysis.class_direction_similarity(Xte, yte, v, NUM_CLASSES)
sim_grad = [torch.nn.functional.cosine_similarity(v.flatten(),
            analysis.gradient_toward(model, Xte[:500], c).flatten(), dim=0).item()
            for c in range(NUM_CLASSES)]
counts = torch.bincount(torch.as_tensor(train.labels).view(-1), minlength=NUM_CLASSES)

table = pd.DataFrame({
    "train images": counts.numpy(),
    "runner-up %": (runner / runner.sum() * 100).numpy(),
    "mean logit margin": margins,
    "cos(v, class mean)": sim_mean.numpy(),
    "cos(v, grad log p_c)": sim_grad,
}, index=short)
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, col in zip(axes, ["runner-up %", "mean logit margin", "cos(v, class mean)", "cos(v, grad log p_c)"]):
    ax.bar(range(NUM_CLASSES), table[col], color="#3b6ea5")
    ax.set_xticks(range(NUM_CLASSES), short, rotation=40, ha="right", fontsize=7)
    ax.set_title(col, fontsize=10)
plt.tight_layout(); plt.savefig("attractor.png", bbox_inches="tight"); plt.show()

**How to read it.** If the attractor class has the highest *runner-up %* and the smallest *margin*, it is simply the
nearest decision region. If instead *cos(v, class mean)* or *cos(v, grad log p_c)* is clearly highest for it, the
perturbation is "painting" features of that class onto the image. Both can be true.

In [ ]:
runs.to_csv("attack_seeds.csv", index=False)
purif.to_csv("purification.csv", index=False)
det.to_csv("detection_auc.csv")
table.to_csv("attractor.csv")